In [ ]:
import pandas as pd
import xarray as xr
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import os
import datetime as dt
import subprocess
from tropycal import tracks

In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-10 18:00", "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        13.7, 14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -33.2, -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ],
    "pressure": [
        1009, 1009, 1009, 1008, 1007, 1006, 1005, 1000, 999, 998, 997, 996, 995,
        994, 991, 985, 980, 975, 970, 966, 962, 960, 956, 955, 958, 959, 958,
        957, 955, 958, 964, 969, 973, 980, 988, 994, 1001, 1004, 1006, 1008, 1010
    ],
    "wind": [
        25, 25, 25, 30, 30, 35, 35, 40, 40, 40, 45, 45, 50, 55, 60, 70, 75, 80,
        85, 90, 95, 95, 100, 105, 95, 85, 85, 80, 80, 75, 70, 70, 65, 60, 55,
        45, 40, 35, 30, 30, 25
    ],
    "stage": [
        "low", "low", "low", "tropical depression", "tropical depression",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "tropical storm", "tropical storm", "tropical storm", "low", "low",
        "low", "low", "low"
    ]
}


In [ ]:
# ============================================================
# --- RADIAL LIGHTNING HOVMOLLER for Hurricane Edouard (2014)
# --- 20-km radial bins, hourly flash density (per 10,000 km²)
# --- Scaled for WWLLN detection efficiency
# --- Clean axes, inverted y-axis
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob

# ---------------- PARAMETERS ----------------
km_per_deg = 111.0
radial_bin_km = 20.0
radial_bin_deg = radial_bin_km / km_per_deg
max_radius_km = 333.0
max_radius_deg = max_radius_km / km_per_deg
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")
detection_efficiency = 0.05  # scale WWLLN data

# ---------------- STORM TRACK ----------------
df_track = pd.DataFrame(track_data)
df_track["datetime"] = pd.to_datetime(df_track["datetime"])
df_track = df_track.set_index("datetime")

interp_times = pd.date_range(start=df_track.index.min(),
                             end=df_track.index.max(),
                             freq="1H")

numeric_cols = ["lat", "lon", "wind", "pressure"]
track_interp = df_track[numeric_cols].reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate(method="time")
track_interp = track_interp.loc[
    (track_interp.index >= time_start) &
    (track_interp.index <= time_end)
]
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()
time_series = track_interp.index

# ---------------- LIGHTNING DATA ----------------
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(
    df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f"
)

# ---------------- RADIAL BINS ----------------
r_edges = np.arange(0, max_radius_deg + radial_bin_deg, radial_bin_deg)
r_centers = (r_edges[:-1] + r_edges[1:]) / 2
LT = np.zeros((len(time_series), len(r_centers)))

# ---------------- BIN LIGHTNING ----------------
for ti, t in enumerate(time_series):
    slat = track_interp.loc[t, "lat"]
    slon = track_interp.loc[t, "lon"]

    t0, t1 = t - pd.Timedelta(minutes=30), t + pd.Timedelta(minutes=30)
    df_sub = df_light_all[
        (df_light_all["datetime"] >= t0) &
        (df_light_all["datetime"] <= t1)
    ]
    if df_sub.empty:
        continue

    dx = (df_sub["longitude"].values - slon) * np.cos(np.deg2rad(slat))
    dy = df_sub["latitude"].values - slat
    r = np.sqrt(dx**2 + dy**2)
    mask = r <= max_radius_deg
    r = r[mask]
    if len(r) == 0:
        continue

    bin_ids = np.digitize(r, r_edges) - 1
    for bi in bin_ids:
        if 0 <= bi < len(r_centers):
            LT[ti, bi] += 1

# ---------------- FLASH DENSITY ----------------
flash_density = np.zeros_like(LT, dtype=float)
for j in range(len(r_centers)):
    r1 = r_edges[j] * km_per_deg
    r2 = r_edges[j+1] * km_per_deg
    area = np.pi * (r2**2 - r1**2)
    flash_density[:, j] = LT[:, j] / area * 1e4  # per 10,000 km²

# Scale for detection efficiency
flash_density_corrected = flash_density / detection_efficiency
flash_plot = np.ma.masked_less(flash_density_corrected, 20)

# ---------------- PLOTTING ----------------
fig, ax = plt.subplots(figsize=(12,6))

# Shading
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Contours at powers of 2
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# ---------------- AXIS FORMATTING ----------------
# ---------------- AXIS FORMATTING ----------------
# Invert Y-axis
ax.set_ylim(r_centers[-1] * km_per_deg, r_centers[0] * km_per_deg)

# Keep tick marks visible, remove numeric labels
ax.tick_params(axis='both', which='both', labelleft=True, labelbottom=True, length=5)

# Make sure spines are visible
for spine in ax.spines.values():
    spine.set_visible(True)

# X-axis limits
ax.set_xlim(time_start, time_end)

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

#plt.tight_layout()
#plt.show()


In [ ]:
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

fig, ax = plt.subplots(figsize=(12,6))

# ---------------- SHADING ----------------
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Contours at powers of 2
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_centers * km_per_deg,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# ---------------- AXES FORMATTING ----------------
# ---------------- AXES FORMATTING ----------------
ax.set_ylabel("Radius from center (km)", labelpad=1)

# Custom y-axis positions for 90 -> 0 -> 270 -> 180 -> 90
# Map these angular-like positions to your radial bins
# We'll use r_centers scaled to km
y_labels = [90, 0, 270, 180, 90]

# Choose radial bins that match the positions (for simplicity, evenly spaced along r_centers)
yticks_pos = np.linspace(r_centers[0]*km_per_deg, r_centers[-1]*km_per_deg, num=len(y_labels))

ax.set_yticks(yticks_pos)
ax.set_yticklabels(y_labels)

# Invert y-axis so 90 starts at the bottom
#ax.invert_yaxis()


ax.set_xlim(time_start, time_end)
ax.set_title(
    f"Lightning Radius–Time Hovmöller\n{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

#plt.tight_layout()
#plt.show()


In [ ]:
import os, glob, subprocess, datetime as dt
import pandas as pd, numpy as np, xarray as xr, matplotlib.pyplot as plt
from matplotlib.patches import Circle
from tropycal import tracks

# --- Time range ---
start_cutoff, end_cutoff = pd.Timestamp("2014-09-11"), pd.Timestamp("2014-09-18 23:59")

# --- Track data ---
df_track = pd.DataFrame(track_data).assign(datetime=lambda x: pd.to_datetime(x["datetime"])).set_index("datetime")
interp_times = pd.date_range(df_track.index.min(), df_track.index.max(), freq="30min")
track_interp = df_track.reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate("time").loc[start_cutoff:end_cutoff]
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()

# --- Output folder ---
os.makedirs("outputs_edouard_frames", exist_ok=True)

# --- Datasets ---
ds_merg = xr.open_mfdataset(sorted(glob.glob("/spare11/atm350/es348865/MERGIR_data/merg_201409*.nc4")), combine="by_coords")
df_light_all = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv"))])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f")

# --- SHIPS shear ---
storm = tracks.TrackDataset(basin="north_atlantic").get_storm(("edouard", 2014))
ships = storm.get_ships(dt.datetime(2014, 9, 11, 12))
times = [ships.attrs["forecast_init"] + dt.timedelta(hours=int(h)) for h in ships.fhr]
df_ships = pd.DataFrame({
    "datetime": times,
    "u_shear": -ships.shear_kt * np.sin(np.deg2rad(ships.shear_dir)),
    "v_shear": -ships.shear_kt * np.cos(np.deg2rad(ships.shear_dir)),
    "shear_mag": ships.shear_kt
}).set_index("datetime")

# --- Interpolated shear ---
df_ships_interp = pd.DataFrame(index=interp_times)
for col in ["u_shear", "v_shear", "shear_mag"]:
    df_ships_interp[col] = np.interp(interp_times.view(int), df_ships.index.view(int), df_ships[col])


In [ ]:
# --- FIG. 5-style TB vs. shear-relative azimuth time series (full week, detailed comments) ---


########################            3 degrees from center, fetched every 30 minutes               ###########################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from tqdm import tqdm

# ---------------- PARAMETERS ----------------
inner_radius_deg = 3.0          # Radial distance (degrees) around the storm center to average TB
azimuth_res_deg = 1.0           # Azimuthal bin size (degrees)
fixed_shear_heading = 8.0       # Shear heading (°) relative to which azimuth is defined
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")  # Full week to plot

# ---------------- SELECT TIMES ----------------
# Only include track times within the requested time window
times = track_interp.index[(track_interp.index >= time_start) & (track_interp.index <= time_end)]
if len(times) == 0:
    raise ValueError("No times in track_interp within requested range.")
print(f"Processing {len(times)} time steps for full week.")

# ---------------- AZIMUTH BINS ----------------
# Define edges and centers of azimuth bins for averaging TB
az_edges = np.arange(0, 360 + azimuth_res_deg, azimuth_res_deg)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2.0

# ---------------- SPATIAL SUBSET ----------------
# Restrict the TB dataset to a region covering the full track ±5° to improve computation speed
lat_min, lat_max = track_interp["lat"].min() - 5, track_interp["lat"].max() + 5
lon_min, lon_max = track_interp["lon"].min() - 5, track_interp["lon"].max() + 5
ds_merg_small = ds_merg.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

# ---------------- PRECOMPUTE AZIMUTH/RADIAL GRIDS ----------------
# For each storm time, calculate the azimuth (degrees from north) and radial distance (deg) for every grid point
lons, lats = ds_merg_small["lon"].values, ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

az_all, r_all = {}, {}
for storm_time in tqdm(times, desc="Precomputing az/r grids"):
    lat, lon = float(track_interp.loc[storm_time, "lat"]), float(track_interp.loc[storm_time, "lon"])
    lat_rad = np.deg2rad(lat)
    dx = (lon2d - lon) * np.cos(lat_rad)  # longitude scaled by latitude
    dy = (lat2d - lat)                     # latitude difference
    az_all[storm_time] = (np.degrees(np.arctan2(dx, dy)) + 360) % 360  # azimuth in degrees
    r_all[storm_time] = np.sqrt(dx**2 + dy**2)                          # radial distance

# ---------------- CONTAINER FOR TB MEANS ----------------
# Will store TB averaged over inner_radius for each azimuth bin at each time
tb_time_az = np.full((len(times), len(az_centers)), np.nan)

# ---------------- MAIN LOOP: COMPUTE AZIMUTHAL TB MEANS ----------------
for ti, storm_time in enumerate(tqdm(times, desc="Computing TB azimuth means")):
    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values  # grab TB at nearest time

    az_grid = az_all[storm_time]
    r_grid = r_all[storm_time]

    # Only include points within the inner radius
    mask_inner = r_grid <= inner_radius_deg
    rel_az = (az_grid - fixed_shear_heading) % 360  # shear-relative azimuth
    tb_vals = tb[mask_inner]

    # Bin TB values by azimuth and compute mean
    az_bin_idx = np.digitize(rel_az[mask_inner], az_edges) - 1
    for ai in range(len(az_centers)):
        vals = tb_vals[az_bin_idx == ai]
        if vals.size > 0:
            tb_time_az[ti, ai] = np.nanmean(vals)

# ---------------- PREPARE TIME EDGES FOR PLOTTING ----------------
# Pcolormesh requires edges (one more than centers)
time_array = np.array(times)
time_edges = pd.date_range(start=time_start, end=time_end + pd.Timedelta(minutes=30), freq="30min")
time_edges_num = mdates.date2num(time_edges)  # convert to matplotlib numeric date format

# ---------------- FIGURE ----------------
fig, ax = plt.subplots(figsize=(12, 5))

# Plot TB as a function of time (x) and shear-relative azimuth (y)
pcm = ax.pcolormesh(
    time_edges_num,
    az_edges,
    tb_time_az.T,
    cmap="turbo",
    vmin=180, vmax=300,
    shading="auto"
)

# Colorbar
cbar = fig.colorbar(pcm, ax=ax, pad=0.02, label="Brightness Temperature (K)")

# ---------------- AXES FORMATTING ----------------
ax.set_ylabel("Shear-relative azimuth (°)\n(0° = down-shear)", labelpad=1)
ax.set_ylim(0, 360)
ax.set_yticks(np.arange(0, 361, 45))
ax.set_xlim(time_start, time_end)
ax.set_title(f"Inner-3° TB vs. Shear-Relative Azimuth\n{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}", y=1.15)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily vertical separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

# ---------------- SHEAR QUADRANTS (DSR → DSL → USL → USR) ----------------
quad_ranges = {"DSR": (0, 90), "DSL": (90, 180), "USL": (180, 270), "USR": (270, 360)}
for label, (a0, a1) in quad_ranges.items():
    center_ang = (a0 + a1) / 2.0
    ax.text(time_edges_num[0] - (time_edges_num[-1] - time_edges_num[0]) * 0.01,
            center_ang, label, ha="right", va="center", fontsize=9,
            fontweight="bold", bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))

# ---------------- SHEAR MAGNITUDE INSET ----------------
shear_mag_series = df_ships_interp.loc[time_array, "shear_mag"].reindex(time_array, fill_value=np.nan).values
ax_inset = fig.add_axes([0.1, 0.95, 0.7, 0.05], sharex=ax)
ax_inset.plot(time_array, shear_mag_series, lw=1.2)
ax_inset.set_ylabel("Shear (kt)", fontsize=8)
ax_inset.grid(True, ls=":", lw=0.5)
ax_inset.xaxis.set_visible(False)
ax_inset.set_ylim(np.nanmin(shear_mag_series) - 2, np.nanmax(shear_mag_series) + 2)

plt.subplots_adjust(bottom=0.12, top=0.9, left=0.1, right=0.95, hspace=0.3)
plt.show()


In [ ]:
################################              NEW CODE HERE

In [ ]:
# ----------------------------------------------------------
# IR BRIGHTNESS TEMPERATURE (Tb) — RADIUS vs TIME HOVMÖLLER
# Matches the same times and storm-track from azimuth plot
# ----------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

# ----------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------
inner_radius_deg = 3.0       # max radius (degrees) from center
radial_bin_deg = 0.05        # radial bin width (degrees)
km_per_deg = 111.0           # constant conversion factor

# Use the *same time window* you already defined
tb_times = times             # from your azimuth script (track_interp range)

# Spatial grid from MERG subset dataset (same as before)
lons = ds_merg_small["lon"].values
lats = ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

# ----------------------------------------------------------
# RADIAL BINS
# ----------------------------------------------------------
r_edges = np.arange(0, inner_radius_deg + radial_bin_deg, radial_bin_deg)
r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
nr = len(r_centers)

# Container for mean Tb: time × radius
tb_time_r = np.full((len(tb_times), nr), np.nan)

# ----------------------------------------------------------
# MAIN LOOP: FOR EACH TIME STEP
# ----------------------------------------------------------
for ti, storm_time in enumerate(tb_times):

    # Storm center at this time
    lat0 = float(track_interp.loc[storm_time, "lat"])
    lon0 = float(track_interp.loc[storm_time, "lon"])

    # Fetch IR Tb field
    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values

    # Compute distance grid (degrees)
    lat_rad = np.deg2rad(lat0)
    dx = (lon2d - lon0) * np.cos(lat_rad)
    dy = (lat2d - lat0)
    r_grid = np.sqrt(dx**2 + dy**2)

    # Mask only the inner region
    mask_inner = r_grid <= inner_radius_deg
    r_vals = r_grid[mask_inner]
    tb_vals = tb[mask_inner]

    # Bin by radius
    r_bin_idx = np.digitize(r_vals, r_edges) - 1

    for ri in range(nr):
        vals = tb_vals[r_bin_idx == ri]
        if vals.size > 0:
            tb_time_r[ti, ri] = np.nanmean(vals)

# ----------------------------------------------------------
# PREP FOR PLOTTING
# ----------------------------------------------------------
time_nums = mdates.date2num(tb_times)
time_edges = np.append(time_nums, time_nums[-1] + (time_nums[-1] - time_nums[-2]))

# Convert radius edges to km
r_edges_km = r_edges * km_per_deg

# ----------------------------------------------------------
# PLOT HOVMÖLLER
# ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

pcm = ax.pcolormesh(
    time_edges,
    r_edges_km,
    tb_time_r.T,
    cmap="turbo",
    vmin=180, vmax=300,
    shading="auto"
)

# Colorbar
cbar = fig.colorbar(pcm, ax=ax, label="IR Brightness Temperature (K)", pad=0.02)

# Axis formatting
ax.set_ylabel("Radius from center (km)", fontsize=12)
ax.set_xlabel("Time (UTC)", fontsize=12)
ax.set_ylim(0, inner_radius_deg * km_per_deg)

ax.set_xlim(tb_times[0], tb_times[-1])
ax.xaxis.set_major_formatter(DateFormatter("%m-%d\n%H:%M"))
plt.title("IR Brightness Temperature — Radius vs Time", fontsize=14)

# Daily separators (matching your style)
for day in pd.date_range(tb_times[0].normalize(),
                         tb_times[-1].normalize() + pd.Timedelta(days=1),
                         freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

fig, ax = plt.subplots(figsize=(12, 6))

# ============================================================
# 1. BUILD THE CORRECT RADIAL AXIS (17 BINS)
# ============================================================
num_times, num_rbins = flash_plot.shape      # (169, 17)
# radius in km (adjust the "3" if your lightning domain uses a different outer radius)
r_km = np.linspace(0, 3 * km_per_deg, num_rbins)

# ============================================================
# 2. SHADED FILLED CONTOURS (FLASH DENSITY)
# ============================================================
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

pcm = ax.contourf(
    mdates.date2num(time_series),  # x-axis length 169
    r_km,                           # y-axis length 17
    flash_plot.T,                   # plot is (17 × 169)
    levels=shade_levels,
    colors=shade_colors,
    extend="max"
)

# ============================================================
# 3. WHITE CONTOUR LINES (POWERS OF 2)
# ============================================================
contour_levels = [20, 40, 80, 160, 320, 640]

ax.contour(
    mdates.date2num(time_series),
    r_km,
    flash_plot.T,
    levels=contour_levels,
    colors="white",
    linewidths=0.8
)

# ============================================================
# 4. AXES FORMATTING
# ============================================================
ax.set_ylabel("Radius from center (km)", labelpad=1)
ax.set_xlim(time_start, time_end)

ax.set_title(
    f"Lightning Radius–Time Hovmöller\n"
    f"{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)

ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# ============================================================
# 5. DAILY SEPARATORS
# ============================================================
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
flash_plot.shape
len(r_centers)
len(time_series)


In [ ]:
flash_plot.shape


In [ ]:
import pandas as pd

# Load RMW data (header=1 because column names are in the second row)
rmw_df = pd.read_csv("../ANIMATIONS_BRIGHTNESS/RMW Edouard - Sheet2.csv", header=1)

# Check columns
print(rmw_df.columns)

# Convert to datetime using the 'date' and 'time' columns
rmw_df['datetime'] = pd.to_datetime(rmw_df['date'] + ' ' + rmw_df['time'], errors='coerce')

# Set datetime as index
rmw_df = rmw_df.set_index('datetime')

# Keep only the RMW column in km
rmw_series = rmw_df['km']
print(rmw_series.head())


In [ ]:
import pandas as pd

# Load CSV (header row 1)
rmw_df = pd.read_csv("../ANIMATIONS_BRIGHTNESS/RMW Edouard - Sheet2.csv", header=1)

# Strip whitespace from column names
rmw_df.columns = rmw_df.columns.str.strip()

# Inspect columns
print(rmw_df.columns)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import pandas as pd

# ------------------------------------------------------------
# Load RMW data
# ------------------------------------------------------------
rmw_df = pd.read_csv("../ANIMATIONS_BRIGHTNESS/RMW Edouard - Sheet2.csv", header=1)
rmw_df['datetime'] = pd.to_datetime(rmw_df['date'] + ' ' + rmw_df['time'], errors='coerce')
rmw_df = rmw_df.set_index('datetime')
rmw_series = rmw_df['km']

# Interpolate RMW to match your lightning time series
rmw_interp = rmw_series.reindex(time_series, method='nearest')  # or use .interpolate('time') for smoother

# ------------------------------------------------------------
# Create plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

# 1. BUILD THE CORRECT RADIAL AXIS
num_times, num_rbins = flash_plot.shape
r_km = np.linspace(0, 3 * km_per_deg, num_rbins)

# 2. SHADED FILLED CONTOURS (FLASH DENSITY)
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]

ax.contourf(
    mdates.date2num(time_series),
    r_km,
    flash_plot.T,
    levels=shade_levels,
    colors=shade_colors,
    extend="max"
)

# 3. WHITE CONTOUR LINES
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_km,
    flash_plot.T,
    levels=contour_levels,
    colors="white",
    linewidths=0.8
)

# 4. OVERLAY RMW LINE
ax.plot(
    mdates.date2num(time_series),
    rmw_interp.values,
    color='red',
    linewidth=2,
    label='RMW (km)'
)
ax.legend(loc='upper right')

# 5. AXES FORMATTING
ax.set_ylabel("Radius from center (km)", labelpad=1)
ax.set_xlim(time_start, time_end)
ax.set_title(
    f"Lightning Radius–Time Hovmöller with RMW\n"
    f"{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# 6. DAILY SEPARATORS
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

# -----------------------------------------------------------
# 1. Load RMW data
# -----------------------------------------------------------
rmw_df = pd.read_csv("../ANIMATIONS_BRIGHTNESS/RMW Edouard - Sheet2.csv", header=1)
rmw_df['datetime'] = pd.to_datetime(rmw_df['date'] + ' ' + rmw_df['time'], errors='coerce')
rmw_df = rmw_df.set_index('datetime')
rmw_series = rmw_df['km']

# -----------------------------------------------------------
# 2. Interpolate RMW to match time_series
# -----------------------------------------------------------
rmw_interp = rmw_series.reindex(time_series, method='nearest')

# -----------------------------------------------------------
# 3. Build radial axis
# -----------------------------------------------------------
num_times, num_rbins = flash_plot.shape      # e.g., (169,17)
r_km = np.linspace(0, 3 * km_per_deg, num_rbins)

# Create 2D grids
r_2d, t_2d = np.meshgrid(r_km, np.arange(num_times), indexing='ij')  # shape (17,169)
r_2d = r_2d.T  # now (169,17), matches flash_plot
rmw_2d = np.tile(rmw_interp.values[:, np.newaxis], (1, num_rbins))  # shape (169,17)

# Mask flashes below RMW for pink shading
flash_below_rmw = np.ma.masked_where(r_2d > rmw_2d, flash_plot)

# Mask flashes above RMW for normal shading
flash_above_rmw = np.ma.masked_where(r_2d <= rmw_2d, flash_plot)

# -----------------------------------------------------------
# 4. Plotting
# -----------------------------------------------------------
fig, ax = plt.subplots(figsize=(12,6))

# Shaded contours above RMW (gray/black)
shade_levels = [20, 80, 160, 320, 1e6]
shade_colors = ["#d9d9d9", "#969696", "#525252", "#000000"]
ax.contourf(
    mdates.date2num(time_series),
    r_km,
    flash_above_rmw.T,
    levels=shade_levels,
    colors=shade_colors,
    extend='max'
)

# Shaded contours below RMW (pink)
pink_levels = [0, 20, 80, 160, 320, 1e6]
pink_colors = ["#ffcce6", "#ff99cc", "#ff66b3", "#ff3399", "#ff0066"]
ax.contourf(
    mdates.date2num(time_series),
    r_km,
    flash_below_rmw.T,
    levels=pink_levels,
    colors=pink_colors,
    extend='max'
)

# Overlay lightning contour lines
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    mdates.date2num(time_series),
    r_km,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# Overlay RMW line
ax.plot(
    mdates.date2num(time_series),
    rmw_interp.values,
    color='red',
    linewidth=2,
    label='RMW'
)

# -----------------------------------------------------------
# 5. Axes formatting
# -----------------------------------------------------------
ax.set_ylabel("Radius from center (km)", labelpad=1)
ax.set_xlabel("Time (UTC)")
ax.set_xlim(time_start, time_end)
ax.set_title(
    f"Lightning Radius–Time Hovmöller with RMW Highlight\n"
    f"{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
from matplotlib.colors import ListedColormap

# -----------------------------
# Load and interpolate RMW
# -----------------------------
rmw_df = pd.read_csv("../ANIMATIONS_BRIGHTNESS/RMW Edouard - Sheet2.csv", header=1)
rmw_df['datetime'] = pd.to_datetime(rmw_df['date'] + ' ' + rmw_df['time'], errors='coerce')
rmw_df = rmw_df.set_index('datetime')
rmw_series = rmw_df['km']
rmw_interp = rmw_series.reindex(time_series, method='nearest')

num_times, num_rbins = flash_plot.shape
r_km = np.linspace(0, 3 * km_per_deg, num_rbins)

# -----------------------------
# Create 2D grids
# -----------------------------
r_2d, t_2d = np.meshgrid(r_km, np.arange(num_times))  # (num_times, num_rbins)
rmw_2d = np.tile(rmw_interp.values[:, np.newaxis], (1, num_rbins))

# -----------------------------
# Numeric mask: 1 = below RMW (pink), 0 = above RMW (gray)
# -----------------------------
mask_numeric = np.zeros_like(flash_plot, dtype=float)
mask_numeric[r_2d <= rmw_2d] = 1.0

# -----------------------------
# Colormap for mask
# -----------------------------
cmap_mask = ListedColormap(['#d9d9d9', 'pink'])

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(12,6))

# pcolormesh needs (time, radius) meshgrid
time_nums = mdates.date2num(time_series)
time_2d, r_2d_plot = np.meshgrid(time_nums, r_km, indexing='ij')

# Plot pink/gray background
pcm = ax.pcolormesh(time_2d, r_2d_plot, mask_numeric, cmap=cmap_mask, shading='auto')

# Overlay lightning contours
contour_levels = [20, 40, 80, 160, 320, 640]
ax.contour(
    time_nums,
    r_km,
    flash_plot.T,
    levels=contour_levels,
    colors='white',
    linewidths=0.8
)

# Overlay RMW line
ax.plot(time_nums, rmw_interp.values, color='red', linewidth=2, label='RMW')

# Axes formatting
ax.set_ylabel("Radius from center (km)")
ax.set_xlabel("Time (UTC)")
ax.set_xlim(time_start, time_end)
ax.set_title(
    f"Lightning Radius–Time Hovmöller with RMW Highlight\n"
    f"{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.05
)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

plt.legend()
plt.tight_layout()
plt.show()
